In [1]:
import pandas as pd
import numpy as np
import random
import re

In [2]:
!pip install -q huggingface_hub

In [3]:
from google.colab import userdata
from huggingface_hub import login
token = userdata.get("HF_Token")
login(token)

In [4]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6a4b8800e645f043c8200847', 'name': 'aniyah839', 'fullname': 'Aniyah McWilliams', 'isPro': False, 'avatarUrl': '/avatars/53a377771ddb857392b3332431b1934e.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'hackathon_colab', 'role': 'fineGrained', 'createdAt': '2026-07-06T16:06:21.976Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '6a4b8800e645f043c8200847', 'type': 'user', 'name': 'aniyah839'}, 'permissions': ['repo.access.read', 'repo.content.read', 'repo.write', 'collection.read', 'collection.write']}]}}}}


# Going to attempt to load the data from Github


In [6]:
!git clone https://github.com/aniyahlater/ds-6051-hackathon

fatal: destination path 'ds-6051-hackathon' already exists and is not an empty directory.


In [7]:
%cd ds-6051-hackathon

/content/ds-6051-hackathon


In [8]:
! git pull https://github.com/aniyahlater/ds-6051-hackathon

remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 33 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (33/33), 947.96 KiB | 16.34 MiB/s, done.
From https://github.com/aniyahlater/ds-6051-hackathon
 * branch            HEAD       -> FETCH_HEAD
Updating 1ab22dc..240281c
Fast-forward
 truthfulqa/avg_avoids_misconception.png            | Bin 0 -> 54421 bytes
 truthfulqa/avg_clarity.png                         | Bin 0 -> 46876 bytes
 truthfulqa/avg_factual_accuracy.png                | Bin 0 -> 52032 bytes
 truthfulqa/category_difficulty_analysis.csv        |  49 ++
 truthfulqa/category_performance.png                | Bin 0 -> 223889 bytes
 truthfulqa/example_responses.csv                   |  11 +
 truthfulqa/failure_analysis.csv                    |   5 +
 truthfulqa/model_improvement_table.csv             |   6 +
 truthfulqa/pass_rate.png                       

In [9]:
df = pd.read_parquet("data/openbookqa.parquet")

In [10]:
df.head()

,id,question_stem,choices,answerKey
0,7-980,The sun is responsible for,"{'text': ['puppies learning new tricks', 'chil...",D
1,7-584,When standing miles away from Mount Rushmore,"{'text': ['the mountains seem very close', 'th...",D
2,7-870,When food is reduced in the stomach,"{'text': ['the mind needs time to digest', 'ta...",C
3,7-321,Stars are,"{'text': ['warm lights that float', 'made out ...",C
4,9-732,You can make a telescope with a,"{'text': ['straw', 'Glass', 'Candle', 'mailing...",D


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-4-E2B-it"  # use exact HF name if different

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [12]:
# def generate_student_answer(question, choices):
#     messages = [
#         {
#             "role": "user",
#             "content": f"""
# You are a student taking a multiple-choice test.

# Question:
# {question}

# Choices:
# {choices}

# Reply with ONLY one letter: A, B, C, or D.
# """
#         }
#     ]

#     inputs = tokenizer.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         return_tensors="pt",
#         return_dict=True
#     ).to(model.device)

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=5,
#         do_sample=False,
#         pad_token_id=tokenizer.eos_token_id
#     )

#     input_length = inputs["input_ids"].shape[-1]

#     response = tokenizer.decode(
#         outputs[0][input_length:],
#         skip_special_tokens=True
#     ).strip().upper()

#     # Keep only a valid letter
#     for letter in ["A", "B", "C", "D"]:
#         if letter == response or response.startswith(letter):
#             return letter

#     return "A"  # Fallback

In [13]:
def ask_gemma(question, choices, answer_key, student_answer, max_new_tokens=200):
    messages = [
        {
            "role": "user",
            "content": f"""
You are an educational assessment grader.

Your job is to evaluate whether the student's answer is correct.

Question:
{question}

Choices:
{choices}

Correct Answer:
{answer_key}

Student Answer:
{student_answer}

Respond ONLY in the following format:

Grade: Correct or Incorrect
Explanation: One or two sentences explaining why.
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    input_length = inputs["input_ids"].shape[-1]

    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )

    return response.strip()

In [15]:
import json
import re

def gemma_quality_judge(question, choices, gemma_response, max_new_tokens=300):
    judge_prompt = f"""You are an impartial evaluator for an educational AI tutor.

Evaluate the assistant's response to a multiple-choice question.

Question:
{question}

Choices:
{choices}

Assistant response:
{gemma_response}

Score the response from 1 to 5 for each category:

1. format_adherence:
5 = follows the exact "Answer: <letter>" / "Explanation: ..." format requested.
1 = ignores the requested format entirely.

2. explanation_quality:
5 = explanation is accurate, relevant, and clearly justifies the chosen answer.
1 = explanation is missing, irrelevant, or nonsensical.

3. politeness:
5 = respectful, supportive, and encouraging in tone.
1 = rude, dismissive, or judgmental.

Return ONLY valid JSON in this exact format, with no extra text before or after:
{{
  "format_adherence": 0,
  "explanation_quality": 0,
  "politeness": 0,
  "reason": "brief explanation"
}}"""

    messages = [
        {"role": "user", "content": judge_prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    input_len = inputs["input_ids"].shape[-1]
    judge_text = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    return parse_judge_json(judge_text)


def parse_judge_json(judge_text):
    """Extract the JSON block from the judge's raw output, tolerating
    stray text or markdown fences around it."""
    match = re.search(r"\{.*\}", judge_text, re.DOTALL)
    if not match:
        return {
            "format_adherence": None,
            "explanation_quality": None,
            "politeness": None,
            "reason": "PARSE_FAILED",
            "raw_judge_text": judge_text,
        }

    try:
        parsed = json.loads(match.group(0))
        parsed["raw_judge_text"] = judge_text
        return parsed
    except json.JSONDecodeError:
        return {
            "format_adherence": None,
            "explanation_quality": None,
            "politeness": None,
            "reason": "PARSE_FAILED",
            "raw_judge_text": judge_text,
        }

In [16]:
import re
import ast
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def generate_student_answer(question, choices, max_new_tokens=150):
    messages = [
        {
            "role": "user",
            "content": f"""Multiple choice question:

Question:
{question}

Choices:
{choices}

Select the best answer.

Respond in this exact format:

Answer: <letter>
Explanation: <one sentence>"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return response


def extract_answer(response):
    match = re.search(r"Answer:\s*([A-D])", response, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return "UNKNOWN"


results = []
correct = 0
subset = df.sample(n=10, random_state=42)

for _, row in subset.iterrows():
    question = row["question_stem"]
    print("\nQuestion:")
    print(question)

    raw_choices = row["choices"]
    if isinstance(raw_choices, str):
        raw_choices = ast.literal_eval(raw_choices)

    labels = raw_choices["label"]
    texts = raw_choices["text"]

    formatted_choices_lines = []
    for label, text in zip(labels, texts):
        formatted_choices_lines.append(f"{label}. {text}")
        print(f"{label}. {text}")

    formatted_choices = "\n".join(formatted_choices_lines)
    answer_key = row["answerKey"]

    gemma_response = generate_student_answer(question, formatted_choices)
    print("\nGemma response:")
    print(gemma_response)

    student_answer = extract_answer(gemma_response)
    print(f"\nGemma selected: {student_answer}")
    print(f"Correct answer: {answer_key}")

    gemma_response = generate_student_answer(question, formatted_choices)
    student_answer = extract_answer(gemma_response)
    is_correct = student_answer == answer_key
    if is_correct:
      correct += 1

    judge_scores = gemma_quality_judge(question, formatted_choices, gemma_response)

    results.append({
    "id": row["id"],
    "question": question,
    "student_answer": student_answer,
    "correct_answer": answer_key,
    "correct": is_correct,
    "gemma_response": gemma_response,
    "format_adherence": judge_scores.get("format_adherence"),
    "explanation_quality": judge_scores.get("explanation_quality"),
    "politeness": judge_scores.get("politeness"),
    "judge_reason": judge_scores.get("reason"),
})

total = len(subset)
accuracy = correct / total

print("\n------------------------")
print(f"Score: {correct}/{total}")
print(f"Accuracy: {accuracy:.1%}")
print("------------------------")

results_df = pd.DataFrame(results)

print("\nAverage quality scores:")
print(results_df[["format_adherence", "explanation_quality", "politeness"]].mean())


display(results_df)
results_df.to_csv("grading_results.csv", index=False)


Question:
Which is likeliest to metamorphose?
A. a live insect
B. a human
C. a plant
D. a dead butterfly

Gemma response:
Answer: A
Explanation: Live insects have the highest potential for undergoing metamorphosis into a different life stage (e.g., larva to pupa to adult).

Gemma selected: A
Correct answer: A

Question:
Which animal is most likely to eat another living animal?
A. deer
B. elephant
C. worm
D. lion

Gemma response:
Answer: D
Explanation: Lions are apex predators known for hunting and eating other large animals.

Gemma selected: D
Correct answer: D

Question:
If a see through thing is multifaceted, it is most likely
A. a ball
B. an apple
C. a silver globe
D. a quartz square

Gemma response:
Answer: C
Explanation: A silver globe is inherently multifaceted due to its spherical shape and reflective surface.

Gemma selected: C
Correct answer: D

Question:
Which two objects would electricity best flow through?
A. a tin can and a plastic fork
B. a steel beam and a soda can
C. a

,id,question,student_answer,correct_answer,correct,gemma_response,format_adherence,explanation_quality,politeness,judge_reason
0,12-878,Which is likeliest to metamorphose?,A,A,True,Answer: A\nExplanation: Live insects have the ...,5,4,5,"The answer is correct, and the explanation acc..."
1,11-287,Which animal is most likely to eat another liv...,D,D,True,Answer: D\nExplanation: Lions are apex predato...,5,5,5,The response correctly identifies the lion as ...
2,13-886,"If a see through thing is multifaceted, it is ...",C,D,False,Answer: C\nExplanation: A silver globe is inhe...,5,4,5,The answer is correct (a sphere/globe can be c...
3,11-497,Which two objects would electricity best flow ...,B,B,True,Answer: B\nExplanation: Electricity flows best...,5,5,5,The response correctly identifies 'steel beam ...
4,9-173,When a wolf is buried what will happen to othe...,D,C,False,Answer: D\nExplanation: The burial of one wolf...,5,4,5,The format was followed perfectly. The explana...
5,1014,An animal that may like a banana peel is a,B,B,True,Answer: B\nExplanation: Raccoons are known for...,5,5,5,The response correctly identifies the correct ...
6,14-788,What is made of minerals?,A,A,True,Answer: A\nExplanation: Stonehenge is a struct...,5,4,5,"The answer is correct, the explanation accurat..."
7,12-352,"If you find an animal that isnt breathing, it ...",C,C,True,Answer: C\nExplanation: An animal that is not ...,5,5,5,The response correctly identifies 'perished' a...
8,14-350,Which is true?,C,C,True,Answer: C\nExplanation: Hot coffee can cause b...,5,5,5,The response correctly identifies option C as ...
9,8-24,"A hurricane is growing on the east coast, and ...",C,C,True,Answer: C\nExplanation: Hurricanes build up du...,5,5,5,The assistant correctly identified option C as...


# Comparing to E2B

In [21]:
model_id = "google/gemma-4-E2B"  # use exact HF name if different

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True
)

tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}{{ '<start_of_turn>user\n' + message['content'] + '<end_of_turn>\n' }}{% elif message['role'] == 'system' %}{{ '<start_of_turn>system\n' + message['content'] + '<end_of_turn>\n' }}{% elif message['role'] == 'model' %}{{ '<start_of_turn>model\n' + message['content'] + '<end_of_turn>\n' }}{% endif %}{% endfor %}"


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [35]:
import re

def generate_student_answer(question, choices, max_new_tokens=50):
    messages = [
        {
            "role": "user",
            "content": f"""Multiple choice question:

Question:
{question}

Choices:
{choices}

Select the best answer.

Respond in this exact format:

Answer: <letter>
Explanation: <one sentence>"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return response


def extract_answer(response):
    match = re.search(r"Answer:\s*([A-D])", response, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return "UNKNOWN"

In [39]:
model_id = "google/gemma-4-E2B"  # keeping base model as specified

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True
)
# No chat_template override — base models weren't trained on turn tokens,
# so plain-prompt completion is more reliable than a hand-rolled chat format.


def generate_student_answer(question, choices, max_new_tokens=100):
    prompt = f"""Multiple choice question:

Question:
{question}

Choices:
{choices}

Select the best answer.

Answer: """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return response


def extract_answer(response):
    match = re.search(r"\b([A-D])\b", response.upper())
    if match:
        return match.group(1)
    return "UNKNOWN"


def ask_gemma(question, choices, answer_key, student_answer, max_new_tokens=50):
    prompt = f"""Grade this multiple choice question.

Question:
{question}

Choices:
{choices}

Correct answer:
{answer_key}

Student answer:
{student_answer}

Grade (Correct or Incorrect), with a one sentence explanation:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return response.strip()


def gemma_quality_judge(question, choices, gemma_response, max_new_tokens=150):
    judge_prompt = f"""Evaluate the following AI response to a multiple choice question.

Question:
{question}

Choices:
{choices}

Assistant response:
{gemma_response}

Score each category from 1 to 5:

format_adherence: does the response give a single clear answer letter?
1 = no clear letter given, 5 = clear single letter given

explanation_quality: is there a relevant explanation for the choice?
1 = no explanation or irrelevant, 5 = clear relevant explanation

politeness: is the tone respectful and neutral?
1 = rude or dismissive, 5 = respectful and clear

Return ONLY JSON in this exact format:
{{"format_adherence": 0, "explanation_quality": 0, "politeness": 0, "reason": "brief explanation"}}

JSON:
"""

    inputs = tokenizer(judge_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    judge_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return parse_judge_json(judge_text)


def parse_judge_json(judge_text):
    match = re.search(r"\{.*\}", judge_text, re.DOTALL)
    if not match:
        return {
            "format_adherence": None,
            "explanation_quality": None,
            "politeness": None,
            "reason": "PARSE_FAILED",
            "raw_judge_text": judge_text,
        }
    try:
        parsed = json.loads(match.group(0))
        parsed["raw_judge_text"] = judge_text
        return parsed
    except json.JSONDecodeError:
        return {
            "format_adherence": None,
            "explanation_quality": None,
            "politeness": None,
            "reason": "PARSE_FAILED",
            "raw_judge_text": judge_text,
        }


results = []
correct = 0
subset = df.sample(n=5, random_state=42)

for _, row in subset.iterrows():
    question = row["question_stem"]
    print(question)

    raw_choices = row["choices"]
    if isinstance(raw_choices, str):
        raw_choices = ast.literal_eval(raw_choices)
    labels = raw_choices["label"]
    texts = raw_choices["text"]

    formatted_choices_lines = []
    for label, text in zip(labels, texts):
        formatted_choices_lines.append(f"{label}. {text}")
        print(f"{label}. {text}")
    formatted_choices = "\n".join(formatted_choices_lines)

    answer_key = row["answerKey"]

    gemma_response = generate_student_answer(question, formatted_choices)
    student_answer = extract_answer(gemma_response)

    print("Gemma response:")
    print(gemma_response)
    print("Gemma answer:")
    print(student_answer)

    is_correct = student_answer == answer_key
    if is_correct:
        correct += 1

    grading_response = ask_gemma(
        question=question,
        choices=formatted_choices,
        answer_key=answer_key,
        student_answer=student_answer
    )
    print(grading_response)

    judge_scores = gemma_quality_judge(question, formatted_choices, gemma_response)

    results.append({
        "id": row["id"],
        "question": question,
        "student_answer": student_answer,
        "correct_answer": answer_key,
        "correct": is_correct,
        "gemma_response": gemma_response,
        "format_adherence": judge_scores.get("format_adherence"),
        "explanation_quality": judge_scores.get("explanation_quality"),
        "politeness": judge_scores.get("politeness"),
        "judge_reason": judge_scores.get("reason"),
    })

total = len(subset)
accuracy = correct / total
print(f"\nScore: {correct}/{total}")
print(f"Accuracy: {accuracy:.1%}")

results_df = pd.DataFrame(results)
print("\nAverage quality scores:")
print(results_df[["format_adherence", "explanation_quality", "politeness"]].mean())

display(results_df)
results_df.to_csv("grading_results.csv", index=False)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Which is likeliest to metamorphose?
A. a live insect
B. a human
C. a plant
D. a dead butterfly
Gemma response:
<strong>a living organism</strong> (the correct option)


Explanation: The metamorphosis of an animal, such as butterflies and moths or insects like grasshoppers into adults occurs through four stages known collectively as complete transformation; these are egg stage larva pupal/pupa adult
Gemma answer:
A
Incorrect, because the correct response was A and not B as stated in my book
Which animal is most likely to eat another living animal?
A. deer
B. elephant
C. worm
D. lion
Gemma response:
<strong>(d)</strong> Lion


Explanation: The correct option for this multiple-choice quiz on animals and their eating habits would be (4). Lions are known as apex predators, meaning they occupy a top position in an ecosystem's food chain or web of life where no other species can feed directly upon them; thus making it more probable that lions will consume any available prey including those st

,id,question,student_answer,correct_answer,correct,gemma_response,format_adherence,explanation_quality,politeness,judge_reason
0,12-878,Which is likeliest to metamorphose?,A,A,True,<strong>a living organism</strong> (the correc...,None,None,None,PARSE_FAILED
1,11-287,Which animal is most likely to eat another liv...,D,D,True,<strong>(d)</strong> Lion\n\n\nExplanation: Th...,None,None,None,PARSE_FAILED
2,13-886,"If a see through thing is multifaceted, it is ...",A,D,False,<strong>a</strong>\n\n\nExplanation of Answer ...,None,None,None,PARSE_FAILED
3,11-497,Which two objects would electricity best flow ...,UNKNOWN,B,False,<strong>human skin</strong> & <strong>rubber g...,None,None,None,PARSE_FAILED
4,9-173,When a wolf is buried what will happen to othe...,B,C,False,<strong>more</strong> (of)\n\n\nExplanation/Re...,None,None,None,PARSE_FAILED


In [41]:
! cd "/content/ds-6051-hackathon/openbookqa"

In [42]:
! pwd

/content/ds-6051-hackathon
